In [2]:
from pathlib import Path


files = ["sample",] #  "input"

connections = {}

for file in files:
    with Path(file).open() as input:
        for line in input.readlines():
            source = line[:line.find(":")]
            destinations = line[line.find(":") + 1:].strip().split()
            connections[source] = destinations
    

In [3]:
connections

{'aaa': ['you', 'hhh'],
 'you': ['bbb', 'ccc'],
 'bbb': ['ddd', 'eee'],
 'ccc': ['ddd', 'eee', 'fff'],
 'ddd': ['ggg'],
 'eee': ['out'],
 'fff': ['out'],
 'ggg': ['out'],
 'hhh': ['ccc', 'fff', 'iii'],
 'iii': ['out']}

In [ ]:
def follow(source: str, connections: dict[str], path:list[str]):
    num_paths: int = 0
    if len(connections[source]) == 1 and connections[source][0] == "out":
        num_paths += 1
    else:
        for d in connections[source]:
            path.append(d)
            num_paths += follow(d, connections, path.copy())

    return num_paths


follow("you", connections, [])

5

In [7]:
from pathlib import Path


files = [
        "sample", 
         "input",]  #  "input"

connections = {}

for file in files:
    with Path(file).open() as input:
        for line in input.readlines():
            source, _, destinations = line.partition(":")
            destinations = destinations.strip().split()
            connections[source] = destinations

    num_paths = follow("you", connections, [])
    print(f"{file=}, {num_paths=}")

file='sample', num_paths=5
file='input', num_paths=539


Improved with ChatGPT

In [10]:
%%timeit

from pathlib import Path
from typing import Dict, List

def load_connections(files: List[str]) -> Dict[str, List[str]]:
    """Load connection mapping from a list of files."""
    connections: Dict[str, List[str]] = {}
    for file in files:
        path = Path(file)
        if not path.exists():
            print(f"Warning: {file} does not exist")
            continue

        with path.open() as f:
            for line in f:
                line = line.strip()
                if not line or ":" not in line:
                    continue
                source, _, dest_str = line.partition(":")
                destinations = dest_str.strip().split()
                connections[source] = destinations
    return connections


def follow(source: str, connections: Dict[str, List[str]]) -> int:
    """Recursively count paths from source to 'out'."""
    if connections.get(source) == ["out"]:
        return 1

    num_paths = 0
    for dest in connections.get(source, []):
        num_paths += follow(dest, connections)
    return num_paths


if __name__ == "__main__":
    files = ["sample", "input"]
    connections = load_connections(files)

    for file in files:
        num_paths = follow("you", connections)
        # print(f"{file=}, {num_paths=}")


408 μs ± 6.74 μs per loop (mean ± std. dev. of 7 runs, 1,000 loops each)


Optimized with ChatGPT (using cache from functools (which I've used 2 years ago for AoC as well))

In [13]:
%%timeit

from pathlib import Path
from typing import Dict, List
from functools import cache  # Python 3.9+; for older versions use lru_cache

def load_connections(files: List[str]) -> Dict[str, List[str]]:
    """Load connection mapping from a list of files."""
    connections: Dict[str, List[str]] = {}
    for file in files:
        path = Path(file)
        if not path.exists():
            print(f"Warning: {file} does not exist")
            continue

        with path.open() as f:
            for line in f:
                line = line.strip()
                if not line or ":" not in line:
                    continue
                source, _, dest_str = line.partition(":")
                destinations = dest_str.strip().split()
                connections[source] = destinations
    return connections


def follow_optimized(connections: Dict[str, List[str]]) -> int:
    """Return a function to count paths from a source with memoization."""
    
    @cache
    def dfs(source: str) -> int:
        if connections.get(source) == ["out"]:
            return 1
        return sum(dfs(dest) for dest in connections.get(source, []))
    
    return dfs


if __name__ == "__main__":
    files = ["sample", "input"]
    connections = load_connections(files)

    dfs = follow_optimized(connections)
    
    for file in files:
        num_paths = dfs("you")
        # print(f"{file=}, {num_paths=}")


238 μs ± 2.32 μs per loop (mean ± std. dev. of 7 runs, 1,000 loops each)


# Part 2

Find number of Paths from "srv" to "out" that visit both "fft" and "dac".

In [ ]:
from pathlib import Path
correct_path_count = 0

def follow_track(source: str, connections: dict[str], path: list[str]):
    global correct_path_count

    if len(connections[source]) == 1 and connections[source][0] == "out":
        if "fft" in path and "dac" in path:
            correct_path_count += 1
            print(path)
            if(correct_path_count % 4 == 0):
                print(f"{correct_path_count=}")
                return

        # yield path
    else:
        for d in connections[source]:
            path.append(d)
            follow_track(d, connections, path.copy())

In [ ]:
files = [
    # "sample2",
    "input",
]  # "input"

connections = {}

for file in files:
    with Path(file).open() as input:
        for line in input.readlines():
            source = line[: line.find(":")]
            destinations = line[line.find(":") + 1 :].strip().split()
            connections[source] = destinations

    correct_path_count = 0
    follow_track("svr", connections, [])
    print(correct_path_count)

In [16]:
from pathlib import Path
from typing import Dict, List

def load_connections(files: List[str]) -> Dict[str, List[str]]:
    """Load connection mapping from a list of files."""
    connections: Dict[str, List[str]] = {}
    for file in files:
        path = Path(file)
        if not path.exists():
            print(f"Warning: {file} does not exist")
            continue
        with path.open() as f:
            for line in f:
                line = line.strip()
                if not line or ":" not in line:
                    continue
                source, _, dest_str = line.partition(":")
                connections[source] = dest_str.strip().split()
    return connections


def follow_track(source: str, connections: Dict[str, List[str]], path: List[str]) -> int:
    """Recursively track paths that include 'fft' and 'dac' and end at 'out'."""
    path.append(source)
    count = 0

    if connections.get(source) == ["out"]:
        if "fft" in path and "dac" in path:
            count = 1
            # Optionally print every 4th path
            if count % 4 == 0:
                print(f"Correct paths so far: {count}")
    else:
        for dest in connections.get(source, []):
            count += follow_track(dest, connections, path)
    
    path.pop()  # backtrack
    return count


if __name__ == "__main__":
    files = ["input"]
    connections = load_connections(files)

    total_correct_paths = follow_track("svr", connections, [])
    print(f"Total correct paths: {total_correct_paths}")


KeyboardInterrupt: 

## Improved / Solved with ChatGPT

In [21]:
from pathlib import Path
from typing import Dict, List
from functools import cache

def load_connections(files: List[str]) -> Dict[str, List[str]]:
    connections: Dict[str, List[str]] = {}
    for file in files:
        path = Path(file)
        if not path.exists():
            print(f"Warning: {file} does not exist")
            continue
        with path.open() as f:
            for line in f:
                line = line.strip()
                if not line or ":" not in line:
                    continue
                source, _, dest_str = line.partition(":")
                connections[source] = dest_str.strip().split()
    return connections


def follow_track_cached(connections: Dict[str, List[str]]) -> int:
    """Return a function to count paths including 'fft' and 'dac' with caching."""
    
    correct_paths = []

    @cache
    def count_paths_from(node: str) -> int:
        """Count valid paths starting from node (memoized)."""
        if connections.get(node) == ["out"]:
            # 'out' itself cannot contain fft/dac, handled in tracking
            return 1
        return sum(count_paths_from(dest) for dest in connections.get(node, []))

    def dfs(node: str, path: List[str]) -> None:
        """DFS to print paths with 'fft' and 'dac' using memoized counts."""
        path.append(node)

        if connections.get(node) == ["out"]:
            if "fft" in path and "dac" in path:
                correct_paths.append(list(path))
                if len(correct_paths) % 1_000_000 == 0:
                    print(f"Correct paths so far: {len(correct_paths)}")

        else:
            for dest in connections.get(node, []):
                dfs(dest, path)
        
        path.pop()  # backtrack

    dfs("svr", [])
    return len(correct_paths)


if __name__ == "__main__":
    files = ["input"]
    connections = load_connections(files)

    total_correct_paths = follow_track_cached(connections)
    print(f"Total correct paths: {total_correct_paths}")


Correct paths so far: 1000000
Correct paths so far: 2000000


KeyboardInterrupt: 

In [25]:
%%timeit

from pathlib import Path
from typing import Dict, List
from functools import cache

def load_connections(files: List[str]) -> Dict[str, List[str]]:
    connections: Dict[str, List[str]] = {}
    for file in files:
        path = Path(file)
        if not path.exists():
            continue
        with path.open() as f:
            for line in f:
                line = line.strip()
                if not line or ":" not in line:
                    continue
                source, _, dest_str = line.partition(":")
                connections[source] = dest_str.strip().split()
    return connections


def count_valid_paths(connections: Dict[str, List[str]], start: str) -> int:
    """
    Count paths from start to 'out' that pass through both 'fft' and 'dac'.
    Avoid building full paths; propagate required flags.
    """

    @cache
    def dfs(node: str, has_fft: bool, has_dac: bool) -> int:
        has_fft |= (node == "fft")
        has_dac |= (node == "dac")

        if connections.get(node) == ["out"]:
            return int(has_fft and has_dac)

        return sum(dfs(dest, has_fft, has_dac) for dest in connections.get(node, []))

    return dfs(start, False, False)


if __name__ == "__main__":
    files = ["input"]
    connections = load_connections(files)
    total_paths = count_valid_paths(connections, "svr")
    # print(f"Total valid paths: {total_paths:,}")


944 μs ± 18.6 μs per loop (mean ± std. dev. of 7 runs, 1,000 loops each)
